# P-023 — Missingness Impact Analysis

P-020 flagged that **Perovskite_thickness** (65% missing) and **Perovskite_band_gap** (31% missing) are imputed to zero via `fillna(0)`, which may inflate tree-based importance by creating artificial split points at 0 vs. non-zero.

**Decision:** How much does the high missingness rate in key features affect model performance and panel rankings?

**Method:** Ablation — train the locked ET config with progressively fewer features, measuring tau-b and panel survival across 10 splits.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('perovskite_stability_clean.csv')

FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'

ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

PANEL = [850, 1320, 119]

# Feature missingness audit
print("=" * 70)
print("FEATURE MISSINGNESS AUDIT")
print("=" * 70)
print(f"{'Feature':<45} {'Missing%':>8} {'Unique':>7}")
print("-" * 70)
miss_info = []
for f in FEATURES:
    pct = df[f].isna().mean() * 100
    n_unique = df[f].dropna().nunique()
    miss_info.append((f, pct, n_unique))

miss_info.sort(key=lambda x: -x[1])
for f, pct, nu in miss_info:
    flag = " <<<" if pct > 10 else ""
    print(f"{f:<45} {pct:>7.1f}% {nu:>7}{flag}")

print(f"\nDataset: {len(df)} devices, {len(FEATURES)} features")
print(f"Features with >10% missing: {sum(1 for _, p, _ in miss_info if p > 10)}")

FEATURE MISSINGNESS AUDIT
Feature                                       Missing%  Unique
----------------------------------------------------------------------
Perovskite_thickness                             65.2%      68 <<<
Perovskite_band_gap                              30.7%      67 <<<
first_Prvskt_thermal_annealing_time               9.0%      36
first_Prvskt_annealing_temperature                7.6%      35
JV_default_FF                                     3.8%     296
JV_default_Voc                                    3.7%     254
JV_default_Jsc                                    3.3%     699
Cell_area_measured                                2.7%     115
Pb                                                0.2%      27
Sn                                                0.2%       9
MA                                                0.1%      63
FA                                                0.1%      60
Cs                                                0.1%      27
I            

In [2]:
# Ablation test: 4 model variants x 10 splits
y = np.log1p(df[TARGET])

# Define feature sets for each variant
high_miss = ['Perovskite_thickness']  # 65% missing
both_miss = ['Perovskite_thickness', 'Perovskite_band_gap']  # 65% + 31%
low_miss_threshold = 10.0  # percent

feat_full = FEATURES[:]
feat_no_thick = [f for f in FEATURES if f != 'Perovskite_thickness']
feat_no_both = [f for f in FEATURES if f not in both_miss]
feat_low_miss = [f for f in FEATURES if df[f].isna().mean() * 100 <= low_miss_threshold]

variants = {
    'Full (16 feat)': feat_full,
    'Drop thickness (15)': feat_no_thick,
    'Drop thick+bandgap (14)': feat_no_both,
    f'Low-miss only ({len(feat_low_miss)})': feat_low_miss,
}

print(f"Low-missingness features ({len(feat_low_miss)}): {feat_low_miss}")
print()

N_SPLITS = 10
results = {}

for name, feats in variants.items():
    tau_list = []
    top20_panel = {d: 0 for d in PANEL}
    
    for seed in range(N_SPLITS):
        X_var = df[feats].fillna(0)
        X_tr, X_te, y_tr, y_te = train_test_split(X_var, y, test_size=0.2, random_state=seed)
        
        model = ExtraTreesRegressor(**ET_PARAMS)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        
        tau, _ = kendalltau(y_te, y_pred)
        tau_list.append(tau)
        
        # Panel check: predict on full dataset, check if panel in top-20
        y_full_pred = model.predict(X_var)
        ranks = pd.Series(y_full_pred).rank(ascending=False)
        for d in PANEL:
            if ranks.iloc[d] <= 20:
                top20_panel[d] += 1
    
    results[name] = {
        'n_features': len(feats),
        'mean_tau': np.mean(tau_list),
        'std_tau': np.std(tau_list),
        'tau_list': tau_list,
        'panel_top20': top20_panel,
    }
    
    print(f"{name:<30} tau-b = {np.mean(tau_list):.4f} +/- {np.std(tau_list):.4f}  "
          f"(features: {len(feats)})")
    for d in PANEL:
        print(f"  Device {d}: in top-20 {top20_panel[d]}/{N_SPLITS} splits")
    print()

Low-missingness features (14): ['Pb', 'Sn', 'I', 'Br', 'Cl', 'MA', 'FA', 'Cs', 'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time', 'Cell_area_measured', 'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF']



Full (16 feat)                 tau-b = 0.3000 +/- 0.0448  (features: 16)
  Device 850: in top-20 9/10 splits
  Device 1320: in top-20 0/10 splits
  Device 119: in top-20 5/10 splits



Drop thickness (15)            tau-b = 0.2840 +/- 0.0442  (features: 15)
  Device 850: in top-20 9/10 splits
  Device 1320: in top-20 0/10 splits
  Device 119: in top-20 5/10 splits



Drop thick+bandgap (14)        tau-b = 0.2884 +/- 0.0460  (features: 14)
  Device 850: in top-20 9/10 splits
  Device 1320: in top-20 0/10 splits
  Device 119: in top-20 7/10 splits



Low-miss only (14)             tau-b = 0.2884 +/- 0.0460  (features: 14)
  Device 850: in top-20 9/10 splits
  Device 1320: in top-20 0/10 splits
  Device 119: in top-20 7/10 splits



In [3]:
# Ablation comparison: how much does dropping high-missingness features change tau-b?
print("=" * 70)
print("ABLATION COMPARISON")
print("=" * 70)

baseline = results['Full (16 feat)']
print(f"\nBaseline (Full model): tau-b = {baseline['mean_tau']:.4f} +/- {baseline['std_tau']:.4f}")
print()

for name, res in results.items():
    if name == 'Full (16 feat)':
        continue
    delta = res['mean_tau'] - baseline['mean_tau']
    pct_change = delta / baseline['mean_tau'] * 100
    print(f"{name}:")
    print(f"  tau-b = {res['mean_tau']:.4f} +/- {res['std_tau']:.4f}")
    print(f"  delta = {delta:+.4f} ({pct_change:+.1f}%)")
    
    if abs(delta) < 0.02:
        print(f"  --> Feature removal has MINIMAL impact (<0.02)")
    elif abs(delta) < 0.05:
        print(f"  --> Feature removal has MODERATE impact (0.02-0.05)")
    else:
        print(f"  --> Feature removal has LARGE impact (>0.05)")
    print()

print("Interpretation:")
print("If tau-b barely drops when removing high-missingness features,")
print("those features aren't contributing real signal despite their MDI importance.")
print("If it drops significantly, they carry real signal despite missingness.")

ABLATION COMPARISON

Baseline (Full model): tau-b = 0.3000 +/- 0.0448

Drop thickness (15):
  tau-b = 0.2840 +/- 0.0442
  delta = -0.0161 (-5.4%)
  --> Feature removal has MINIMAL impact (<0.02)

Drop thick+bandgap (14):
  tau-b = 0.2884 +/- 0.0460
  delta = -0.0117 (-3.9%)
  --> Feature removal has MINIMAL impact (<0.02)

Low-miss only (14):
  tau-b = 0.2884 +/- 0.0460
  delta = -0.0117 (-3.9%)
  --> Feature removal has MINIMAL impact (<0.02)

Interpretation:
If tau-b barely drops when removing high-missingness features,
those features aren't contributing real signal despite their MDI importance.
If it drops significantly, they carry real signal despite missingness.


In [4]:
# Panel survival analysis
print("=" * 70)
print("PANEL SURVIVAL ANALYSIS")
print("=" * 70)
print(f"Panel devices: {PANEL}")
print(f"Criterion: device appears in top-20 predicted stability across 10 splits\n")

for name, res in results.items():
    print(f"{name}:")
    all_survive = True
    for d in PANEL:
        rate = res['panel_top20'][d]
        status = "SURVIVES" if rate >= 5 else "LOST"
        if rate < 5:
            all_survive = False
        print(f"  Device {d}: top-20 in {rate}/{N_SPLITS} splits  [{status}]")
    panel_status = "ALL SURVIVE" if all_survive else "PANEL DISRUPTED"
    print(f"  --> {panel_status}")
    print()

# Compare specifically: does panel survive without thickness? without thickness+bandgap?
print("Summary:")
for name in ['Drop thickness (15)', 'Drop thick+bandgap (14)']:
    res = results[name]
    survived = sum(1 for d in PANEL if res['panel_top20'][d] >= 5)
    print(f"  {name}: {survived}/{len(PANEL)} panel devices survive in top-20")

PANEL SURVIVAL ANALYSIS
Panel devices: [850, 1320, 119]
Criterion: device appears in top-20 predicted stability across 10 splits

Full (16 feat):
  Device 850: top-20 in 9/10 splits  [SURVIVES]
  Device 1320: top-20 in 0/10 splits  [LOST]
  Device 119: top-20 in 5/10 splits  [SURVIVES]
  --> PANEL DISRUPTED

Drop thickness (15):
  Device 850: top-20 in 9/10 splits  [SURVIVES]
  Device 1320: top-20 in 0/10 splits  [LOST]
  Device 119: top-20 in 5/10 splits  [SURVIVES]
  --> PANEL DISRUPTED

Drop thick+bandgap (14):
  Device 850: top-20 in 9/10 splits  [SURVIVES]
  Device 1320: top-20 in 0/10 splits  [LOST]
  Device 119: top-20 in 7/10 splits  [SURVIVES]
  --> PANEL DISRUPTED

Low-miss only (14):
  Device 850: top-20 in 9/10 splits  [SURVIVES]
  Device 1320: top-20 in 0/10 splits  [LOST]
  Device 119: top-20 in 7/10 splits  [SURVIVES]
  --> PANEL DISRUPTED

Summary:
  Drop thickness (15): 2/3 panel devices survive in top-20
  Drop thick+bandgap (14): 2/3 panel devices survive in top-20


In [5]:
# Save Packet_P023_Missingness_Impact.csv
rows = []
for name, res in results.items():
    for seed_idx, tau_val in enumerate(res['tau_list']):
        row = {
            'variant': name,
            'n_features': res['n_features'],
            'seed': seed_idx,
            'tau_b': tau_val,
            'mean_tau_b': res['mean_tau'],
            'std_tau_b': res['std_tau'],
        }
        for d in PANEL:
            row[f'device_{d}_top20_rate'] = res['panel_top20'][d] / N_SPLITS
        rows.append(row)

out_df = pd.DataFrame(rows)
out_df.to_csv('Packet_P023_Missingness_Impact.csv', index=False)
print(f"Saved Packet_P023_Missingness_Impact.csv: {len(out_df)} rows "
      f"({len(results)} variants x {N_SPLITS} splits)")
print(out_df.groupby('variant')[['tau_b']].mean().sort_values('tau_b', ascending=False))

Saved Packet_P023_Missingness_Impact.csv: 40 rows (4 variants x 10 splits)
                            tau_b
variant                          
Full (16 feat)           0.300027
Drop thick+bandgap (14)  0.288361
Low-miss only (14)       0.288361
Drop thickness (15)      0.283974


In [6]:
# Status footer
baseline_tau = results['Full (16 feat)']['mean_tau']
drop_thick_tau = results['Drop thickness (15)']['mean_tau']
drop_both_tau = results['Drop thick+bandgap (14)']['mean_tau']

delta_thick = baseline_tau - drop_thick_tau
delta_both = baseline_tau - drop_both_tau

# Panel survival check
panel_survives_no_thick = all(
    results['Drop thickness (15)']['panel_top20'][d] >= 5 for d in PANEL
)
panel_survives_no_both = all(
    results['Drop thick+bandgap (14)']['panel_top20'][d] >= 5 for d in PANEL
)

# Determine status
if delta_both < 0.02 and panel_survives_no_both:
    status = "Confirmed"
    reason = (f"Dropping both high-missingness features loses only {delta_both:.4f} tau-b "
              f"(<0.02 threshold) and panel survives.")
elif delta_both < 0.05:
    status = "Promising"
    reason = (f"Dropping both high-missingness features loses {delta_both:.4f} tau-b "
              f"(between 0.02-0.05). Moderate impact.")
else:
    status = "Negative"
    reason = (f"Dropping both high-missingness features loses {delta_both:.4f} tau-b "
              f"(>0.05). Model depends on imputed features.")

print("=" * 70)
print(f"P-023 STATUS: {status}")
print("=" * 70)
print(f"\n{reason}")
print(f"\nDelta tau-b (drop thickness only):    {delta_thick:+.4f}")
print(f"Delta tau-b (drop thick + bandgap):    {delta_both:+.4f}")
print(f"Panel survives without thickness:      {panel_survives_no_thick}")
print(f"Panel survives without thick+bandgap:  {panel_survives_no_both}")
print(f"\nConclusion: High-missingness features {'do NOT materially affect' if delta_both < 0.02 else 'DO affect'} "
      f"model ranking quality.")
print(f"{'Their MDI importance is likely inflated by the imputation artifact.' if delta_both < 0.02 else 'Despite missingness, these features carry real predictive signal.'}")

P-023 STATUS: Promising

Dropping both high-missingness features loses 0.0117 tau-b (between 0.02-0.05). Moderate impact.

Delta tau-b (drop thickness only):    +0.0161
Delta tau-b (drop thick + bandgap):    +0.0117
Panel survives without thickness:      False
Panel survives without thick+bandgap:  False

Conclusion: High-missingness features do NOT materially affect model ranking quality.
Their MDI importance is likely inflated by the imputation artifact.
